In [1]:
%config InlineBackend.figure_format = 'retina'
import os
import sys
import importlib
from pathlib import Path
root_path = Path(os.getcwd())
sys.path.append(str(Path('..').resolve()))

import math
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from loader import load_player_data_for_scoring
from rosters_2026 import rosters_2026
from aggregation import aggregate_all
from steps.filter import filter_all, filtering_summary
from steps.decay import apply_decay, decay_summary
from steps.normalization import normalize_all, normalization_summary
from steps.segmentation import apply_segmentation, segmentation_summary
from steps.scoring import build_scores, scoring_summary

STATSBOMB_DIR = Path("..") / "data" / "Statsbomb"

In [2]:
# Build flat set of all rostered players
import rosters_2026
importlib.reload(rosters_2026)
from rosters_2026 import rosters_2026
rostered_players = {
    player 
    for squad in rosters_2026.values() 
    for player in squad.keys()
}
print(f"Total rostered players: {len(rostered_players)}")

Total rostered players: 469


In [3]:
matches = pl.read_parquet("../data/Statsbomb/matches.parquet")
raw = load_player_data_for_scoring("recent_club_players")
clean = aggregate_all(raw, matches, verbose=False)


Loading: recent_club_players
Seasons: ['2021/2022', '2022/2023', '2023/2024']

  ── 2021/2022 ──
    ✓ xg__player__totals.csv — 293 rows, 284 players
    ✓ progression__player__profile.csv — 1,175 rows, 1123 players
    ✓ advanced__player__packing.csv — 5,104 rows, 1053 players
    ✓ advanced__player__xg_chain.csv — 5,403 rows, 1151 players
    ✓ advanced__player__xg_buildup.csv — 4,731 rows, 1083 players
    ✓ advanced__player__network_centrality.csv — 6,194 rows, 1244 players
    ✓ defensive__player__profile.csv — 1,919 rows, 650 players
    ✓ defensive__player__pressures.csv — 5,647 rows, 1167 players

  ── 2022/2023 ──
    ✓ xg__player__totals.csv — 134 rows, 131 players
    ✓ progression__player__profile.csv — 884 rows, 846 players
    ✓ advanced__player__packing.csv — 2,417 rows, 874 players
    ✓ advanced__player__xg_chain.csv — 2,421 rows, 888 players
    ✓ advanced__player__xg_buildup.csv — 2,125 rows, 822 players
    ✓ advanced__player__network_centrality.csv — 2,740 rows, 9

In [4]:
filtered = filter_all(clean, verbose=False)
filtering_summary(filtered)


FILTERING SUMMARY
  xg__player__totals.csv
    599 rows | 528 players | 82 shrinkage-flagged | seasons: ['2021/2022', '2022', '2022/2023', '2023', '2023/2024', '2024']
  progression__player__profile.csv
    1,524 rows | 1304 players | 512 shrinkage-flagged | seasons: ['2021/2022', '2022/2023', '2023/2024']
  advanced__player__packing.csv
    2,086 rows | 1715 players | 384 shrinkage-flagged | seasons: ['2021/2022', '2022', '2022/2023', '2023', '2023/2024', '2024']
  advanced__player__xg_chain.csv
    1,422 rows | 1216 players | 481 shrinkage-flagged | seasons: ['2021/2022', '2022', '2022/2023', '2023', '2023/2024', '2024']
  advanced__player__xg_buildup.csv
    1,280 rows | 1095 players | 424 shrinkage-flagged | seasons: ['2021/2022', '2022', '2022/2023', '2023', '2023/2024', '2024']
  advanced__player__network_centrality.csv
    1,521 rows | 1302 players | 510 shrinkage-flagged | seasons: ['2021/2022', '2022', '2022/2023', '2023', '2023/2024', '2024']
  defensive__player__profile.csv

In [5]:
decayed = apply_decay(filtered, verbose=False)
decay_summary(decayed)


DECAY SUMMARY
  xg__player__totals.csv
    599 rows | 528 players | weights present: [0.8, 0.9, 1.0]
  progression__player__profile.csv
    1,524 rows | 1304 players | weights present: [0.8, 0.9, 1.0]
  advanced__player__packing.csv
    2,086 rows | 1715 players | weights present: [0.8, 0.9, 1.0]
  advanced__player__xg_chain.csv
    1,422 rows | 1216 players | weights present: [0.8, 0.9, 1.0]
  advanced__player__xg_buildup.csv
    1,280 rows | 1095 players | weights present: [0.8, 0.9, 1.0]
  advanced__player__network_centrality.csv
    1,521 rows | 1302 players | weights present: [0.8, 0.9, 1.0]
  defensive__player__profile.csv
    625 rows | 570 players | weights present: [0.8, 0.9, 1.0]
  defensive__player__pressures.csv
    1,378 rows | 1181 players | weights present: [0.8, 0.9, 1.0]


In [6]:
for filename, df in decayed.items():
    metrics_in_file = [
        (k, v["column"]) 
        for k, v in __import__('player_score.player_metrics_config', fromlist=['PLAYER_METRICS']).PLAYER_METRICS.items()
        if v["file"] == filename and v["column"] in df.columns
    ]
    for metric_key, col in metrics_in_file:
        stats = df[col].drop_nulls()
        print(f"{metric_key}: min={stats.min():.3f}, median={stats.median():.3f}, max={stats.max():.3f}, skew={stats.skew():.2f}")

xg_volume: min=0.070, median=0.880, max=23.300, skew=4.98
progressive_passes: min=0.000, median=1.760, max=8.950, skew=1.12
progressive_carries: min=0.000, median=1.155, max=10.840, skew=1.50
packing: min=0.000, median=0.000, max=2.467, skew=1.16
xg_chain: min=0.009, median=0.422, max=1.814, skew=1.07
team_involvement: min=3.175, median=36.130, max=97.559, skew=0.45
xg_buildup: min=0.007, median=0.305, max=1.849, skew=1.29
network_centrality: min=2.241, median=16.362, max=36.332, skew=0.11
defensive_actions: min=27.000, median=63.000, max=639.000, skew=2.83
high_turnovers: min=0.000, median=5.000, max=87.000, skew=3.11
pressure_volume: min=0.796, median=12.177, max=40.275, skew=0.70
pressure_success: min=0.000, median=25.808, max=79.575, skew=0.59


In [7]:
import steps.normalization
importlib.reload(steps.normalization)
from steps.normalization import *
normalized = normalize_all(decayed, verbose=False)
normalization_summary(normalized)


NORMALIZATION SUMMARY
  xg__player__totals.csv: 2 norm columns
    finishing_quality_norm: min=0.115, median=0.500, max=1.000, nulls=0
    xg_volume_norm: min=0.000, median=0.500, max=1.000, nulls=0
  progression__player__profile.csv: 2 norm columns
    progressive_passes_norm: min=0.054, median=0.500, max=1.000, nulls=0
    progressive_carries_norm: min=0.085, median=0.501, max=1.000, nulls=0
  advanced__player__packing.csv: 1 norm columns
    packing_norm: min=0.000, median=0.500, max=1.000, nulls=0
  advanced__player__xg_chain.csv: 2 norm columns
    xg_chain_norm: min=0.000, median=0.500, max=1.000, nulls=0
    team_involvement_norm: min=0.000, median=0.500, max=1.000, nulls=0
  advanced__player__xg_buildup.csv: 1 norm columns
    xg_buildup_norm: min=0.000, median=0.500, max=1.000, nulls=0
  advanced__player__network_centrality.csv: 1 norm columns
    network_centrality_norm: min=0.000, median=0.500, max=1.000, nulls=0
  defensive__player__profile.csv: 2 norm columns
    defensiv

In [8]:
from player_score.steps.shrinkage import apply_shrinkage, shrinkage_summary

LINEUPS_PATH = Path("..") / "data" / "Statsbomb" / "lineups.parquet"

shrunk = apply_shrinkage(normalized, lineups_path=LINEUPS_PATH, verbose=False)
shrinkage_summary(shrunk)


SHRINKAGE SUMMARY
  xg__player__totals.csv
    599 rows | 82 shrinkage-flagged | archetypes: ['AM', 'CB', 'CM', 'DM', 'FB', 'FW', 'W']
    finishing_quality_norm: min=0.115, median=0.522, max=1.000
    xg_volume_norm: min=0.000, median=0.538, max=1.000
  progression__player__profile.csv
    1,524 rows | 512 shrinkage-flagged | archetypes: ['AM', 'CB', 'CM', 'DM', 'FB', 'FW', 'GK', 'W']
    progressive_passes_norm: min=0.054, median=0.511, max=1.000
    progressive_carries_norm: min=0.085, median=0.495, max=1.000
  advanced__player__packing.csv
    2,086 rows | 384 shrinkage-flagged | archetypes: ['AM', 'CB', 'CM', 'DM', 'FB', 'FW', 'GK', 'W']
    packing_norm: min=0.000, median=0.500, max=1.000
  advanced__player__xg_chain.csv
    1,422 rows | 481 shrinkage-flagged | archetypes: ['AM', 'CB', 'CM', 'DM', 'FB', 'FW', 'GK', 'W']
    xg_chain_norm: min=0.000, median=0.542, max=1.000
    team_involvement_norm: min=0.000, median=0.513, max=1.000
  advanced__player__xg_buildup.csv
    1,280 

In [9]:
df = shrunk["advanced__player__network_centrality.csv"]
print(df.filter(pl.col("position_archetype").is_in(["CM", "DM"]))
      .group_by("position_archetype")
      .agg(pl.len().alias("count"))
      .sort("position_archetype"))

shape: (2, 2)
┌────────────────────┬───────┐
│ position_archetype ┆ count │
│ ---                ┆ ---   │
│ str                ┆ u32   │
╞════════════════════╪═══════╡
│ CM                 ┆ 143   │
│ DM                 ┆ 235   │
└────────────────────┴───────┘


In [10]:
segmented = apply_segmentation(shrunk, verbose=False)
segmentation_summary(segmented)


SEGMENTATION SUMMARY

  CB — Centre-Back
    advanced__player__network_centrality.csv: 330 rows
    advanced__player__packing.csv: 445 rows
    advanced__player__xg_buildup.csv: 303 rows
    advanced__player__xg_chain.csv: 310 rows
    defensive__player__pressures.csv: 329 rows
    defensive__player__profile.csv: 75 rows
    progression__player__profile.csv: 122 rows
    xg__player__totals.csv: 50 rows

  FB — Fullback / Wing-Back
    advanced__player__network_centrality.csv: 286 rows
    advanced__player__packing.csv: 446 rows
    advanced__player__xg_buildup.csv: 247 rows
    advanced__player__xg_chain.csv: 267 rows
    defensive__player__pressures.csv: 284 rows
    defensive__player__profile.csv: 150 rows
    progression__player__profile.csv: 104 rows
    xg__player__totals.csv: 53 rows

  DM — Defensive Midfielder
    advanced__player__network_centrality.csv: 235 rows
    advanced__player__packing.csv: 378 rows
    advanced__player__xg_buildup.csv: 204 rows
    advanced__player__x

In [11]:
import steps.scoring
importlib.reload(steps.scoring)
from steps.scoring import * 

scored = build_scores(segmented, verbose=True)
scoring_summary(scored, top_n=50)


STEP 7 & 8: SCORING + COMPOSITE ASSEMBLY
  [1/4] Building unified player-season table...
         5,679 player-season rows, 1,780 unique players
  [2/4] Collapsing seasons with decay weighting...
         1,780 players after collapse
  [4/4] Computing position-weighted category scores...
         Mobility_Intensity: 449 players, median=49.4
         Progression: 465 players, median=44.1
         Control: 437 players, median=52.3
         Final_Third_Output: 528 players, median=47.8

  Composite score: 1,780 players
  min=6.4, median=32.7, max=83.8

TOP 50 PLAYERS — COMPOSITE SCORE
                                  player                 team position_archetype  composite_score  Mobility_Intensity_score  Progression_score  Control_score  Final_Third_Output_score
rank                                                                                                                                                                                   
1                   Kylian Mbappé Lottin   

In [12]:
events = pd.read_parquet("../data/Statsbomb/events.parquet")
matches = pd.read_parquet("../data/Statsbomb/matches.parquet")
sb_players = set(events["player"].dropna().unique())

matched = rostered_players & sb_players
unmatched = rostered_players - sb_players

print(f"Matched: {len(matched)}")
print(f"Unmatched: {len(unmatched)}")
print("\nSample unmatched:", list(unmatched)[:20])

Matched: 397
Unmatched: 72

Sample unmatched: ['André Onana Onana', 'Saud Abdullah Abdulhamid', 'Joel Ordóñez', 'Amad Diallo Traoré', 'Luiz Henrique', 'Joachim Christian Andersen', 'Oleksandr Volodymyrovych Zinchenko', 'Ahmed Sayed Mohamed', 'Alessandro Buongiorno', 'Bruno Miguel Rivotti Varela', 'Sébastien Romain Teddy Haller', 'Percy Muzi Tau', 'Samuel Essende', 'Ali Hadi Mohammed Al-Bulaihi', 'Vincent Aboubakar', 'Mahmoud Ahmed Ibrahim Hassan', 'Nawaf Al-Aqidi', 'Pau Cubarsí Paredes ', 'Nicolás Paz Martínez', 'Mykhailo Petrovych Mudryk']


In [13]:
from rapidfuzz import process, fuzz

sb_players_list = list(sb_players)

def find_best_match(name, candidates, threshold=80):
    result = process.extractOne(
        name, candidates, 
        scorer=fuzz.token_sort_ratio,
        score_cutoff=threshold
    )
    return result[0] if result else None

# Build mapping: roster_name → statsbomb_name
name_mapping = {}
unresolved = []

for name in unmatched:
    match = find_best_match(name, sb_players_list)
    if match:
        name_mapping[name] = match
    else:
        unresolved.append(name)

print(f"Fuzzy matched: {len(name_mapping)}")
print(f"Still unresolved: {len(unresolved)}")
print("\nSample mappings (verify these):")
for k, v in list(name_mapping.items())[:20]:
    print(f"  '{k}':'{v}'")

Fuzzy matched: 20
Still unresolved: 52

Sample mappings (verify these):
  'Vincent Aboubakar':'Vincent Paté Aboubakar'
  'Mahmoud Ahmed Ibrahim Hassan':'Mahmoud Ibrahim Hassan'
  'Nicolás Paz Martínez':'Nicolás Martín Pareja'
  'Timi Max Elšnik':'Timi Elšnik'
  'Firas Tariq Nasser Al-Buraikan':'Firas Tariq Nasser Al Albirakan'
  'Son Heung-min':'Heung-Min Son'
  'Willi Orbán':'Willi Orban'
  'Ronwen Hayden Williams':'Ronwen Williams'
  'Hwang Hee-chan':'Hee-Chan Hwang'
  'Lee Kang-in':'Kang-In Lee'
  'Thomas Kristos Strakosha':'Thomas Strakosha'
  'Pierre-Emile Kordt Højbjerg':'Pierre-Emile Højbjerg'
  'Alireza Jahanbakhsh Jirandeh':'Alireza Jahanbakhsh'
  'Salem Mohammed Al-Dawsari':'Salem Mohammed Al Dawsari'
  'El Bilal Touré':'El Bilal Toure'
  'Odilon Kossounou':'Odilon Kossonou'
  'Miloš Kerkez':'Milos Kerkez'
  'Mohamed Salah Ghaly':'Mohamed Salah'
  'Amadou Haidara':'Amadou Haïdara'
  'Hwang In-beom':'In-Beom Hwang'


In [14]:
name_mapping_with_scores = {}
for name in unmatched:
    result = process.extractOne(
        name, sb_players_list,
        scorer=fuzz.token_sort_ratio,
        score_cutoff=60  # lower threshold
    )
    if result:
        name_mapping_with_scores[name] = (result[0], result[1])

# Print all with scores for review
for k, (v, score) in sorted(name_mapping_with_scores.items(), key=lambda x: x[1][1], reverse=True):
    print(f"'{k}':'{v}'")

'Odilon Kossounou':'Odilon Kossonou'
'Salem Mohammed Al-Dawsari':'Salem Mohammed Al Dawsari'
'Hwang Hee-chan':'Hee-Chan Hwang'
'El Bilal Touré':'El Bilal Toure'
'Amadou Haidara':'Amadou Haïdara'
'Son Heung-min':'Heung-Min Son'
'Hwang In-beom':'In-Beom Hwang'
'Miloš Kerkez':'Milos Kerkez'
'Willi Orbán':'Willi Orban'
'Lee Kang-in':'Kang-In Lee'
'Mahmoud Ahmed Ibrahim Hassan':'Mahmoud Ibrahim Hassan'
'Pierre-Emile Kordt Højbjerg':'Pierre-Emile Højbjerg'
'Vincent Aboubakar':'Vincent Paté Aboubakar'
'Firas Tariq Nasser Al-Buraikan':'Firas Tariq Nasser Al Albirakan'
'Timi Max Elšnik':'Timi Elšnik'
'Nicolás Paz Martínez':'Nicolás Martín Pareja'
'Mohamed Salah Ghaly':'Mohamed Salah'
'Ronwen Hayden Williams':'Ronwen Williams'
'Alireza Jahanbakhsh Jirandeh':'Alireza Jahanbakhsh'
'Thomas Kristos Strakosha':'Thomas Strakosha'
'Alessandro Buongiorno':'Alessandro Longhi'
'André Onana Onana':'André Onana'
'James Maddison':'James Morrison'
'Percy Muzi Tau':'Percy Tau'
'Saud Abdullah Abdulhamid':'Saud 

In [ ]:
update = {'Odilon Kossounou':'Odilon Kossonou'
'Salem Mohammed Al-Dawsari':'Salem Mohammed Al Dawsari'
'Hwang Hee-chan':'Hee-Chan Hwang'
'El Bilal Touré':'El Bilal Toure'
'Amadou Haidara':'Amadou Haïdara'
'Son Heung-min':'Heung-Min Son'
'Hwang In-beom':'In-Beom Hwang'
'Miloš Kerkez':'Milos Kerkez'
'Willi Orbán':'Willi Orban'
'Lee Kang-in':'Kang-In Lee'
'Mahmoud Ahmed Ibrahim Hassan':'Mahmoud Ibrahim Hassan'
'Pierre-Emile Kordt Højbjerg':'Pierre-Emile Højbjerg'
'Vincent Aboubakar':'Vincent Paté Aboubakar'
'Firas Tariq Nasser Al-Buraikan':'Firas Tariq Nasser Al Albirakan'
'Timi Max Elšnik':'Timi Elšnik'
'Nicolás Paz Martínez':'Nicolás Martín Pareja'
'Mohamed Salah Ghaly':'Mohamed Salah'
'Ronwen Hayden Williams':'Ronwen Williams'
'Alireza Jahanbakhsh Jirandeh':'Alireza Jahanbakhsh'
'Thomas Kristos Strakosha':'Thomas Strakosha'
'André Onana Onana':'André Onana'
'Percy Muzi Tau':'Percy Tau'
'Saud Abdullah Abdulhamid':'Saud Abdullah Abdul Hamid'
'Khuliso Mudau':'Khuliso Johnson Mudau'
'Joachim Christian Andersen':'Joachim Andersen'
'Johnny Cardoso':'Johann Carrasso'
'Mykhailo Petrovych Mudryk':'Mykhailo Mudryk'
'Ahmed Sayed Mohamed':'Ahmed Sayed'
'Oleksandr Volodymyrovych Zinchenko':'Oleksandr Zinchenko'
'Sébastien Romain Teddy Haller':'Sébastien Haller'
'Carlos Noom Quomah Baleba':'Carlos Baleba'}